In [1]:
import pandas as pd
import os
import numpy as np
import folium
import matplotlib.pyplot as plt
import textwrap

from matplotlib.backends.backend_pdf import PdfPages
from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

<div class="alert alert-info">
    <b>Erstellen einer .pdf zu allen gesammelten Datenbanken bzgl. ihres Rock Types</b><br>
    -> Implementierung mit einzelnen .csvs, bevor die Rock-Types über Notebook 1.2 ergänzt wurden
    -> Arbeit erstmal nur mit 1.1
</div>

In [2]:
# ----------------------- Pfad zu Ordner mit .csv Dateien -----------------------
base_dir = Path.cwd()

db_root = (
    base_dir.parent.parent 
    / "1_Acquisition" 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken" / "Nach_Acquisition"
)

# ----------------------- alle .csvs aus einem Ordner sammeln -----------------------
csv_files = sorted(db_root.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Keine .csv-Dateien in {db_root} gefunden.")

# ----------------------- Zielfpad für .pdf -----------------------
output_pdf = base_dir / "Rock-Type_Analysis_without-added-rock-types.pdf"
print(output_pdf)

# ----------------------- Eine Seite pro csv -----------------------
with PdfPages(output_pdf) as pdf:
    for csv_path in csv_files:
        # ----------------------- CSV einlesen -----------------------
        try:
            df = pd.read_csv(csv_path, low_memory=False)
        except Exception as e:
            # ----------------------- Fehlerbehandlung -----------------------
            fig = plt.figure(figsize=(8.27, 11.69))  
            fig.suptitle(f"Datei: {csv_path.name}", fontsize=14, fontweight="bold")
            fig.text(0.1, 0.8, f"Fehler beim Einlesen:\n{e}", fontsize=10)
            plt.axis("off")
            pdf.savefig(fig)
            plt.close(fig)
            continue

        total_rows = len(df)

        # ----------------------- Prüfen ob Rock Type existiert -----------------------
        if "rock_type" in df.columns:
            rock_col = df["rock_type"]
            non_null = rock_col.notna().sum()
            percent = (non_null / total_rows * 100) if total_rows > 0 else 0.0

            # ---------- Einzigartige Werte -----------
            unique_vals = sorted(rock_col.dropna().astype(str).unique())
        else:
            non_null = 0
            percent = 0.0
            unique_vals = []

        # -------------- Text für Seite ---------------
        header_lines = [
            f"Datei: {csv_path.name}",
            "",
            f"Anzahl Zeilen insgesamt: {total_rows}",
            f"Zeilen mit Wert in 'rock_type': {non_null} ({percent:0.2f} %)",
            "",
            "Einzigartige Werte in 'rock_type':"
        ]


        # -------- Einzigartige Werte als Liste in Textform ----------
        if unique_vals:
            # ----------------------- Kommatrennung für Zeichenkette -----------------------
            unique_str = ", ".join(unique_vals)
        else:
            unique_str = "(Keine Werte oder Spalte 'rock_type' nicht vorhanden.)"


        # ----------------------- Zeilen umbrechen, damit es auf die Seite passt -----------------------
        wrapped_unique_lines = textwrap.wrap(unique_str, width=100)

        # ----------------------- Neue Seite erstellen -----------------------
        fig = plt.figure(figsize=(8.27, 11.69))  # -------- A4-Hochformat ----------
        fig.suptitle("Rock-Type-Analyse", fontsize=14, fontweight="bold", y=0.97)

        # ----------------------- Alle Zeilen gesammelt ausgeben -----------------------
        all_lines = header_lines + [""] + wrapped_unique_lines


        # ----------------------- Text von oben nach unten platzieren -----------------------
        y_start = 0.9
        line_height = 0.025

        for i, line in enumerate(all_lines):
            y = y_start - i * line_height
            if y < 0.05:
                fig.text(0.1, 0.05, "... (weitere Werte abgeschnitten)", fontsize=9)
                break
            fig.text(0.1, y, line, fontsize=10, va="top")

        plt.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF erstellt: {output_pdf}")


F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_without-added-rock-types.pdf


PDF erstellt: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.2_Rock-Type_Analysis\Rock-Type_Analysis_without-added-rock-types.pdf
